In [17]:
import flappy_bird_gymnasium
import gymnasium
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import cv2
from collections import deque

In [10]:
env = gymnasium.make('FlappyBird-v0', render_mode='rgb_array', use_lidar=False)

In [11]:
env.observation_space

Box(-1.0, 1.0, (12,), float64)

In [12]:
env.action_space

Discrete(2)

In [19]:
#This cell dedicated to Hyper  parameters

train_ep = 15000
max_steps = 5000
lr = 0.00005
batch_size = 64
max_len = 50000
gamma = 0.99
max_epsilon = 1.0
min_epsilon = 0.05
decay_rate = 0.0003
update_target_network = 2000
step_count = 0
ep_replay = []

In [23]:
#cvtColor to gray out image
#resize to smash img into 80*80 grid

def preprocess_frame(frame):
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    frame = cv2.resize(frame, (80, 80), interpolation=cv2.INTER_AREA)
    return frame

In [39]:
#To stack up 4 frames together to know where it is going and for first 3 frames they will use the the first oen for rem times
class FrameStack:
    def __init__(self):
        self.frames = deque(maxlen = 4)

    def reset(self, initial_frame):
        for _ in range(4):
            self.frames.append(initial_frame)
        return np.stack(self.frames, axis=0)

    def step(self, new_frame):
        self.frames.append(new_frame)
        return np.stack(self.frames, axis=0)

In [42]:
#Experience Replay
class ExperienceReplay:
    def __init__(self, max_len):
        self.memory = deque(maxlen = max_len)

    def add(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.memory, batch_size-2)
        batch.append(self.memory[-1])
        batch.append(self.memory[-2])
        return batch

    def len(self):
        return len(self.memory)

In [24]:
# DQN nitialization using pytorch and dueling dqn
class DQN(nn.Module):
    def __init__(self, input_size = 4, output_size = 2):
        super(DQN, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(input_size, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU()
        )

        #size changes as (initial - kernel_size)/stride + 1
        #size updates: 80 -> 19 -> 8 -> 6
        #64 6X6 images once flattened 
        flattened_size = 64*6*6

        self.value = nn.Sequential(
            nn.Linear(flattened_size, 512),
            nn.ReLU(),
            nn.Linear(512, 1)
        )

        self.advantage = nn.Sequential(
            nn.Linear(flattened_size, 512),
            nn.ReLU(),
            nn.Linear(512, output_size)
        )

    def forward(self, x):
        x = x/255.0
        #first conv stream
        features = self.conv(x)
        #flattening
        features = features.view(features.size(0), -1)
        #dueling streams
        v = self.value(features)
        a = self.advantage(features)

        Q = v + (a - a.mean(dim=1, keepdim=True))

        return Q
            

In [28]:
#initializing policy and target dqn and optimizer
device = 'cuda' if torch.cuda.is_available() else 'cpu'

policy_dqn = DQN().to(device)
target_dqn = DQN().to(device)

target_dqn.load_state_dict(policy_dqn.state_dict())
target_dqn.eval()

optimizer = torch.optim.Adam(policy_dqn.parameters(), lr = lr)

In [26]:
def epsilon_greedy_policy(state, epsilon):
        if random.uniform(0.0, 1.0) < epsilon:
            return env.action_space.sample()

        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)

        with torch.no_grad():
            Qvalues = policy_dqn(state_tensor)

        return Qvalues.argmax().item()

In [43]:
# learning

def learn(batch, gamma):
    states, actions, rewards, next_states, dones = zip(*batch)

    #conversion to pytorch tensors
    states = torch.FloatTensor(np.array(states)).to(device)
    actions = torch.LongTensor(actions).unsqueeze(1).to(device)
    rewards = torch.FloatTensor(rewards).unsqueeze(1).to(device)
    next_states = torch.FloatTensor(np.array(next_states)).to(device)
    dones = torch.FloatTensor(dones).unsqueeze(1).to(device)

    Qvalues_policy = policy_dqn(states).gather(1, actions)

    with torch.no_grad():
        next_actions = policy_dqn(next_states).argmax(dim=1, keepdim=True)
        next_Qvalues = target_dqn(next_states).gather(1, next_actions)

        Qvalues_target = rewards + gamma*next_Qvalues*(1-dones)

    loss = F.smooth_l1_loss(Qvalues_policy, Qvalues_target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [44]:
memory = ExperienceReplay(max_len)
stacker = FrameStack()

> MAIN TRAINING BELOW AND REWARD OUTPUT IS LITTLE BAD I DID NOT REALLY TOOK MEAN AND ALL SO THOSE REWARD OUTPUT DOESN'T REALLLY MEANS ANYTHING MOREOVER I STOPPED TRAINING AT 13K CAUSE IT TOOK 5 HOURS TO IT BUT RESULTS WERE GREAT IN THIS DURATION

In [50]:
#Main Training cell
for i in range(train_ep):
    epsilon = min_epsilon + (max_epsilon - min_epsilon)*np.exp(-decay_rate * i)
    state, info = env.reset()
    ep_reward = 0

    state = env.render()
    state = preprocess_frame(state)
    state = stacker.reset(state)

    for _ in range(max_steps):
        action = epsilon_greedy_policy(state, epsilon)

        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        if(done):
            reward = -10.0
        elif (reward != 1.0):
            reward = 0.05

        ep_reward += reward

        next_state = env.render()
        next_state = preprocess_frame(next_state)
        next_state = stacker.step(next_state)

        memory.add(state, action, reward, next_state, done)

        if memory.len() > batch_size:
            batch = memory.sample(batch_size)
            learn(batch, gamma)

        step_count += 1
        if(step_count >= update_target_network):
            step_count = 0
            target_dqn.load_state_dict(policy_dqn.state_dict())
        
        if(done):
            break

        state = next_state

    if i % 100 == 0:
        print(f"Episode: {i} | Reward: {ep_reward:.2f} | Epsilon: {epsilon:.4f} | Total Steps: {step_count}")

Episode: 0 | Reward: -7.55 | Epsilon: 1.0000 | Total Steps: 100
Episode: 100 | Reward: -7.55 | Epsilon: 0.9719 | Total Steps: 1100
Episode: 200 | Reward: -7.55 | Epsilon: 0.9447 | Total Steps: 100
Episode: 300 | Reward: -7.55 | Epsilon: 0.9182 | Total Steps: 1100
Episode: 400 | Reward: -7.55 | Epsilon: 0.8926 | Total Steps: 100
Episode: 500 | Reward: -7.55 | Epsilon: 0.8677 | Total Steps: 1100
Episode: 600 | Reward: -7.55 | Epsilon: 0.8435 | Total Steps: 100
Episode: 700 | Reward: -7.55 | Epsilon: 0.8201 | Total Steps: 1100
Episode: 800 | Reward: -7.55 | Epsilon: 0.7973 | Total Steps: 100
Episode: 900 | Reward: -7.55 | Epsilon: 0.7752 | Total Steps: 1100
Episode: 1000 | Reward: -7.55 | Epsilon: 0.7538 | Total Steps: 100
Episode: 1100 | Reward: -7.55 | Epsilon: 0.7330 | Total Steps: 1100
Episode: 1200 | Reward: -7.55 | Epsilon: 0.7128 | Total Steps: 100
Episode: 1300 | Reward: -7.55 | Epsilon: 0.6932 | Total Steps: 1100
Episode: 1400 | Reward: -7.55 | Epsilon: 0.6742 | Total Steps: 100


In [51]:
torch.save(policy_dqn.state_dict(), "flappy_bird_brain.pth")

## AI-GENERATED EVALUATION SCRIPT

In [52]:
# --- 1. Define the exact same Brain architecture ---
class DQN(nn.Module):
    def __init__(self, input_size=4, output_size=2):
        super(DQN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(input_size, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU()
        )
        flattened_size = 64 * 6 * 6
        self.value = nn.Sequential(
            nn.Linear(flattened_size, 512),
            nn.ReLU(),
            nn.Linear(512, 1)
        )
        self.advantage = nn.Sequential(
            nn.Linear(flattened_size, 512),
            nn.ReLU(),
            nn.Linear(512, output_size)
        )

    def forward(self, x):
        x = x / 255.0  
        features = self.conv(x)
        features = features.view(features.size(0), -1)
        v = self.value(features)
        a = self.advantage(features)
        Q = v + (a - a.mean(dim=1, keepdim=True))
        return Q

# --- 2. Image Preprocessing & Stacking ---
def preprocess_frame(frame):
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    frame = cv2.resize(frame, (80, 80), interpolation=cv2.INTER_AREA)
    return frame

class FrameStack:
    def __init__(self):
        self.frames = deque(maxlen=4)
    def reset(self, initial_frame):
        for _ in range(4):
            self.frames.append(initial_frame)
        return np.stack(self.frames, axis=0)
    def step(self, new_frame):
        self.frames.append(new_frame)
        return np.stack(self.frames, axis=0)

# --- 3. Setup and Load Weights ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
env = gymnasium.make('FlappyBird-v0', render_mode='rgb_array', use_lidar=False)

# Load the brain
policy_dqn = DQN().to(device)
policy_dqn.load_state_dict(torch.load("flappy_bird_brain.pth", map_location=device))
policy_dqn.eval() # Set to evaluation mode (turns off gradients)

stacker = FrameStack()

# --- 4. Testing Loop ---
test_episodes = 5
total_rewards = []

print("Starting AI Test Phase...")

for episode in range(test_episodes):
    _, info = env.reset()
    raw_state = env.render()
    
    state = preprocess_frame(raw_state)
    state = stacker.reset(state)
    
    ep_reward = 0
    done = False
    
    while not done:
        # Show the game window
        # Convert RGB to BGR for OpenCV display
        bgr_frame = cv2.cvtColor(raw_state, cv2.COLOR_RGB2BGR)
        cv2.imshow("Flappy Bird AI Test", bgr_frame)
        
        # Slow down the frames slightly so you can watch it (30ms delay)
        if cv2.waitKey(30) & 0xFF == ord('q'):
            break

        # AI chooses action (100% exploitation, no epsilon)
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            Qvalues = policy_dqn(state_tensor)
            action = Qvalues.argmax().item()
            
        # Take step
        _, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        # We use the raw environment reward for the final score, not our custom training reward
        ep_reward += reward 
        
        # Prep next frame
        raw_state = env.render()
        next_state = preprocess_frame(raw_state)
        state = stacker.step(next_state)
        
    total_rewards.append(ep_reward)
    print(f"Test Episode {episode + 1} | Score: {ep_reward:.2f}")

# Clean up windows
cv2.destroyAllWindows()
env.close()

# Calculate and print average
avg_reward = sum(total_rewards) / test_episodes
print("-" * 30)
print(f"Testing Complete! Average Score over {test_episodes} episodes: {avg_reward:.2f}")

Starting AI Test Phase...
Test Episode 1 | Score: 373.00
Test Episode 2 | Score: 34.10
Test Episode 3 | Score: 74.90
Test Episode 4 | Score: 424.30
Test Episode 5 | Score: 65.00
------------------------------
Testing Complete! Average Score over 5 episodes: 194.26
